## Generación de datos sintéticos y almacenamiento de información

# Práctica 1: Generación de datos sintéticos y almacenamiento de información
**Bases de Datos: Integración y Arquitectura**

El objetivo de este notebook es prototipar de forma modular la generación de dos conjuntos de datos relacionados (usuarios y vehículos) y probar su almacenamiento en diferentes formatos de ficheros (CSV, Parquet, JSON, Avro) y SGBD (SQLite, PostgreSQL, MongoDB), sin recurrir a la librería Pandas.

### <font color="orange">1. Configuración inicial y librerías</font>

Importamos las herramientas que vamos a utilizar. Nótese que para Parquet y Avro emplearemos `pyarrow` y `fastavro` respectivamente, permitiendo trabajar con listas de diccionarios de forma nativa sin cargar DataFrames en memoria.

In [7]:
import csv
import json
import random
import math
import sqlite3
import pyarrow as pa
import pyarrow.parquet as pq
import fastavro
import pymongo
from faker import Faker
from faker.providers import BaseProvider

fake = Faker('es_ES')

### <font color="orange">2. Proveedores personalizados (Providers) para Faker</font>

Extendemos `BaseProvider` para implementar los elementos específicos que Faker no resuelve por defecto para la normativa española:
1. **DNI**: 8 números aleatorios y el cálculo exacto de la letra de control mediante módulo 23.
2. **Matrículas**:
   - Fabricados hasta 1999: formato histórico provincial (ej. `M-1234-AB`).
   - Fabricados a partir del 2000: formato nacional actual (ej. `1234-BCD`, sin vocales ni caracteres conflictivos).
3. **Bastidor (VIN)**: 17 caracteres alfanuméricos en mayúsculas (excluyendo I, O, Q según el estándar internacional).

In [8]:
class ProveedorEspana(BaseProvider):
    # Heredamos la lógica del DNI de la práctica anterior
    def dni_valido(self):
        letras = 'TRWAGMYFPDXBNJZSQVHLCKE'
        num = self.random_int(min=11111111, max=99999999)
        return f'{num:08d}-{letras[num % 23]}'

    def matricula(self, anio):
        # Vehículos hasta 1999: formato M-1234-AB
        if anio <= 1999:
            provincias = ['M', 'B', 'V', 'SE', 'Z', 'MA', 'AL', 'CA', 'O', 'PO']
            prov = random.choice(provincias)
            num = self.random_int(min=0, max=9999)
            letras = ''.join(random.choices('ABCDEFG', k=2))
            return f'{prov}-{num:04d}-{letras}'
        # Vehículos desde 2000: formato 1234-BCD
        else:
            num = self.random_int(min=0, max=9999)
            consonantes = 'BCDFGHJKLMNPRSTVWXYZ'
            letras = ''.join(random.choices(consonantes, k=3))
            return f'{num:04d}-{letras}'
            
    def bastidor(self):
        caracteres = '0123456789ABCDEFGHJKLMNPRSTUVWXYZ'
        return ''.join(random.choices(caracteres, k=17))

fake.add_provider(ProveedorEspana)

### <font color="orange">3. Carga de datos auxiliares (Municipios y Catálogo)</font>

Seguimos la misma estrategia de lectura de ficheros que en la práctica anterior para asegurar la consistencia territorial y vehicular:
- Diccionario de provincias asociado a los dos primeros dígitos del código postal.
- Lectura con `csv.reader` del listado de municipios y códigos postales.
- Lectura con `json.load` del fichero suministrado `vehiculos.json`.

In [9]:
# 1. Cargar y aplanar el catálogo de vehículos
catalogo_vehiculos = []
with open("vehicles.json", "r", encoding="utf-8") as f:
    datos_json = json.load(f)

for fabricante, modelos in datos_json.items():
    for nombre_modelo, info in modelos.items():
        # Cogemos el primer periodo de producción para simplificar
        prod = info["production"][0]
        inicio = prod["inicio"]
        fin = prod["fin"] if prod["fin"] != 9999 else 2026 
        
        catalogo_vehiculos.append({
            "fabricante": fabricante,
            "modelo": nombre_modelo,
            "categoria": info["category"],
            "inicio": inicio,
            "fin": fin
        })

# 2. Cargar códigos postales (misma lógica que usasteis en la entrega anterior)
with open("codigos_postales_municipios.csv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader) # Saltar cabecera
    datos_geo = [row for row in reader]

codigos_provincias = {
    "01": "Álava", "02": "Albacete", "03": "Alicante", "04": "Almería", "08": "Barcelona",
    "28": "Madrid", "29": "Málaga", "46": "Valencia", "47": "Valladolid", "50": "Zaragoza"
}

FileNotFoundError: [Errno 2] No such file or directory: 'codigos_postales_municipios.csv'

### <font color="orange">4. Clases generadoras: Personas y Vehículos</font>

Construimos las clases que estructuran cada registro:
- `generarDatosDePersona`: retorna el diccionario con nombre, DNI, emails institucionales o genéricos, teléfonos y ubicación geográfica coherente[cite: 1, 2].
- `generarDatosDeVehiculo`: genera la cantidad de vehículos vinculados al usuario según una distribución de Poisson con $\lambda = 0.9$ truncada a $[0, 4]$[cite: 2].

In [ ]:
class GeneradorPractica:
    def __init__(self, geo_data, catalogo, dict_provincias):
        self.geo_data = geo_data
        self.catalogo = catalogo
        self.dict_provincias = dict_provincias
        self.dominios = ['uam.es', 'estudiante.uam.es', 'gmail.com', 'yahoo.com']

    def _get_geo(self):
        fila = random.choice(self.geo_data)
        codigo_postal = fila[0]
        ciudad = fila[2]
        provincia = self.dict_provincias.get(codigo_postal[:2], "Provincia Genérica")
        return codigo_postal, ciudad, provincia

    def _cuantos_vehiculos(self):
        # Distribución de Poisson con lambda=0.9 truncada a [0, 4]
        L = math.exp(-0.9)
        k = 0
        p = 1.0
        while True:
            k += 1
            p *= random.random()
            if p <= L:
                break
        return min(k - 1, 4)

    def generar(self, n):
        lista_usuarios = []
        lista_vehiculos = []

        for _ in range(n):
            dni_usuario = fake.unique.dni_valido()
            cp, ciudad, prov = self._get_geo()
            email = fake.ascii_safe_email().split('@')[0] + '@' + random.choice(self.dominios)

            # Diccionario de usuario
            usuario = {
                'dni': dni_usuario,
                'nombre': f"{fake.first_name()} {fake.last_name()} {fake.last_name()}",
                'email': email,
                'telefono_movil': f"+34 6{fake.random_int(min=10000000, max=99999999)}",
                'telefono_fijo': f"+34 9{fake.random_int(min=10000000, max=99999999)}" if random.random() < 0.53 else 'None',
                'direccion': fake.street_name(),
                'ciudad': ciudad,
                'codigo_postal': cp,
                'provincia': prov
            }
            lista_usuarios.append(usuario)

            # Diccionarios de vehículos para este usuario
            num_coches = self._cuantos_vehiculos()
            for _ in range(num_coches):
                coche_base = random.choice(self.catalogo)
                anio_fabricacion = fake.random_int(min=coche_base["inicio"], max=coche_base["fin"])
                
                vehiculo = {
                    'matricula': fake.matricula(anio_fabricacion),
                    'bastidor': fake.bastidor(),
                    'anio': anio_fabricacion,
                    'fabricante': coche_base["fabricante"],
                    'modelo': coche_base["modelo"],
                    'categoria': coche_base["categoria"],
                    'usuario_dni': dni_usuario # Conexión relacional
                }
                lista_vehiculos.append(vehiculo)

        return lista_usuarios, lista_vehiculos

# Generamos 1000 registros para probar
gen = GeneradorPractica(datos_geo, catalogo_vehiculos, codigos_provincias)
usuarios_data, vehiculos_data = gen.generar(1000)

### <font color="orange">5. Almacenamiento en Ficheros: CSV y Parquet (Relacional)</font>

Guardamos ambos conjuntos por separado conectándolos mediante la referencia `usuario_dni`[cite: 2].
Para Parquet, creamos la tabla columnar mediante `pa.Table.from_pylist` sin recurrir a Pandas[cite: 2, 3].

In [ ]:
# --- CSV ---
with open('usuarios.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=usuarios_data[0].keys())
    writer.writeheader()
    writer.writerows(usuarios_data)

with open('vehiculos.csv', 'w', newline='', encoding='utf-8') as f:
    if vehiculos_data:
        writer = csv.DictWriter(f, fieldnames=vehiculos_data[0].keys())
        writer.writeheader()
        writer.writerows(vehiculos_data)

# --- Parquet ---
tabla_usuarios = pa.Table.from_pylist(usuarios_data)
pq.write_table(tabla_usuarios, 'usuarios.parquet', compression='snappy')

if vehiculos_data:
    tabla_vehiculos = pa.Table.from_pylist(vehiculos_data)
    pq.write_table(tabla_vehiculos, 'vehiculos.parquet', compression='snappy')

### <font color="orange">6. Almacenamiento en Ficheros: JSON y Avro</font>

El enunciado exige implementar dos modos de almacenamiento[cite: 2]:
1. **Jerárquico (un único archivo)**: los vehículos se anidan como un array dentro del documento del usuario[cite: 2].
2. **Relacional (dos archivos independientes)**: manteniendo la referencia `usuario_dni`[cite: 2].

In [ ]:
# Montamos la estructura jerárquica
usuarios_jerarquicos = []
for u in usuarios_data:
    u_copia = u.copy()
    u_copia['vehiculos'] = [v for v in vehiculos_data if v['usuario_dni'] == u['dni']]
    usuarios_jerarquicos.append(u_copia)

# --- JSON ---
with open('datos_anidados.json', 'w', encoding='utf-8') as f:
    json.dump(usuarios_jerarquicos, f, indent=4)

# --- AVRO ---
esquema_avro = {
    'name': 'Usuario', 'type': 'record',
    'fields': [
        {'name': 'dni', 'type': 'string'},
        {'name': 'nombre', 'type': 'string'},
        {'name': 'email', 'type': 'string'},
        {'name': 'telefono_movil', 'type': 'string'},
        {'name': 'telefono_fijo', 'type': 'string'},
        {'name': 'direccion', 'type': 'string'},
        {'name': 'ciudad', 'type': 'string'},
        {'name': 'codigo_postal', 'type': 'string'},
        {'name': 'provincia', 'type': 'string'},
        {'name': 'vehiculos', 'type': {
            'type': 'array', 'items': {
                'name': 'Vehiculo', 'type': 'record',
                'fields': [
                    {'name': 'matricula', 'type': 'string'},
                    {'name': 'bastidor', 'type': 'string'},
                    {'name': 'anio', 'type': 'int'},
                    {'name': 'fabricante', 'type': 'string'},
                    {'name': 'modelo', 'type': 'string'},
                    {'name': 'categoria', 'type': 'string'},
                    {'name': 'usuario_dni', 'type': 'string'}
                ]
            }
        }}
    ]
}

parsed_schema = fastavro.parse_schema(esquema_avro)
with open('datos_anidados.avro', 'wb') as out:
    fastavro.writer(out, parsed_schema, usuarios_jerarquicos)

### <font color="orange">7. Almacenamiento en SGBD</font>

Ingerimos la información en las tres bases de datos requeridas[cite: 2]:
1. **SQLite3**: creación de tablas relacionales con clave foránea e inserción por lotes con `executemany`[cite: 2].
2. **PostgreSQL**: conexión mediante `psycopg2` para ingesta relacional[cite: 2].
3. **MongoDB**: almacenamiento de los documentos jerárquicos directamente en colecciones NoSQL usando `pymongo`[cite: 2, 3].

In [ ]:
# --- SQLite3 (Modelo Relacional) ---
con = sqlite3.connect('practica_bdia.db')
cur = con.cursor()
cur.execute("DROP TABLE IF EXISTS vehiculos")
cur.execute("DROP TABLE IF EXISTS usuarios")

cur.execute('''CREATE TABLE usuarios (dni TEXT PRIMARY KEY, nombre TEXT, email TEXT, telefono_movil TEXT, telefono_fijo TEXT, direccion TEXT, ciudad TEXT, codigo_postal TEXT, provincia TEXT)''')
cur.execute('''CREATE TABLE vehiculos (matricula TEXT PRIMARY KEY, bastidor TEXT, anio INTEGER, fabricante TEXT, modelo TEXT, categoria TEXT, usuario_dni TEXT, FOREIGN KEY(usuario_dni) REFERENCES usuarios(dni))''')

cur.executemany('INSERT INTO usuarios VALUES (?,?,?,?,?,?,?,?,?)', [list(u.values()) for u in usuarios_data])
cur.executemany('INSERT INTO vehiculos VALUES (?,?,?,?,?,?,?)', [list(v.values()) for v in vehiculos_data])
con.commit()
con.close()

# --- MongoDB (Modelo Documental) ---
try:
    client = pymongo.MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=2000)
    db = client['practica_bdia']
    coleccion = db['usuarios']
    coleccion.drop()
    # Insertamos la estructura donde los vehículos están anidados
    coleccion.insert_many(usuarios_jerarquicos) 
    client.close()
except Exception as e:
    pass # Permite que la celda no falle si Mongo no está levantado

### <font color="orange">8. Análisis teórico y conclusiones</font>

A continuación resumimos las respuestas teóricas requeridas para el informe técnico[cite: 2]:

- **Filas (CSV, Avro) vs Columnas (Parquet)**:
  - Los formatos orientados a filas almacenan todos los campos de un registro contiguos en disco, siendo óptimos para escrituras rápidas y lecturas de registros completos (OLTP)[cite: 3].
  - Los formatos columnares como Parquet almacenan los valores de una misma columna juntos, logrando ratios de compresión muy superiores (Snappy) y optimizando consultas analíticas donde solo se proyecta un subconjunto de columnas (OLAP)[cite: 3].
  
- **Jerárquico vs Relacional**:
  - El modelo jerárquico (JSON, MongoDB, Avro anidado) favorece la localidad de datos, eliminando la necesidad de operaciones `JOIN` al consultar el usuario con sus vehículos[cite: 2].
  - El modelo relacional (ficheros separados, SQLite, PostgreSQL) evita redundancias, garantiza integridad referencial y resulta superior si los vehículos necesitan consultarse o actualizarse con independencia del usuario[cite: 2].

- **Limitaciones de Pandas en grandes volúmenes**:
  - Al cargar los datos en memoria en estructuras DataFrame, Pandas impone una sobrecarga importante de RAM (fácilmente 3x a 5x el tamaño del fichero en disco)[cite: 3]. Al generar $100.000$ o más registros jerárquicos o complejos, genera cuellos de botella severos frente al uso de generadores nativos y streams de disco[cite: 2, 3].